# Ordered Logistic Regression Results: FAIR² Dataset Exploration with `mlcroissant`
This notebook provides a complete workflow for loading and exploring the **Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya** dataset using the [`mlcroissant`](https://mlcroissant.org) library.

### Dataset Source
The dataset source is defined by a Croissant schema URL, enabling machine-actionable access to metadata and records.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()

print(f"Dataset Title: {metadata.get('name', '[Unnamed]')}")
print(f"Description: {metadata.get('description', '[No description]')}")
print("\nDataset identifier:", metadata.get('identifier', '[no identifier]'))
print(f"Published: {metadata.get('datePublished', '[no date]')}")

## 2. Data Overview
Review all available record sets, their `@id`s, and contained fields/columns. This is crucial for referencing and loading the correct entities and structures.


In [ ]:
# List all record sets in the dataset, by @id and name
record_sets_info = []
for record_set in dataset.record_sets:
    rs_id = record_set.id
    rs_name = record_set.name
    # List available fields/columns in this record set
    field_ids = [field.id for field in record_set.fields]
    field_names = [field.name for field in record_set.fields]
    record_sets_info.append({'@id': rs_id, 'name': rs_name, 'field_ids': field_ids, 'field_names': field_names})

if not record_sets_info:
    print("No record sets defined in this Croissant package.")
else:
    print("Found the following record sets:")
    for record_set in record_sets_info:
        print(f"Record set: {record_set['name']} (@id: {record_set['@id']})")
        print("  Fields:")
        for fid, fname in zip(record_set['field_ids'], record_set['field_names']):
            print(f"    {fname} (@id: {fid})")
        print("")

## 3. Data Extraction

Extract records for each record set by `@id` and load into Pandas DataFrames using `mlcroissant`. Use the `@id` fields identified above to reference record sets and fields.

In [ ]:
# Collect available record sets by @id
record_set_ids = [rs['@id'] for rs in record_sets_info]

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded record set @id: {record_set_id} with {len(df)} rows and columns: {list(df.columns)}\n")
    else:
        print(f"No records found for record set @id: {record_set_id}")

# Pick one DataFrame for demonstration if present
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"Main record set selected: {main_record_set_id}")
    print("First 5 rows:")
    display(dataframes[main_record_set_id].head())
else:
    main_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Demonstrate basic EDA: filter, normalize, and group data. All fields/columns referenced by their full `@id` as above.

In [ ]:
# EDA: Only proceed if a valid DataFrame is available
from pandas.api.types import is_numeric_dtype
if main_record_set_id is not None:
    df = dataframes[main_record_set_id]
    print(f"Columns in record set {main_record_set_id}:")
    print(list(df.columns))
    # Find a numeric field by @id
    numeric_field_id = None
    group_field_id = None
    for col in df.columns:
        if is_numeric_dtype(df[col]) and col != 'Unnamed: 0':
            numeric_field_id = col
            break
    # Find a group-by-able field (prefer string/categorical)
    for col in df.columns:
        if col != numeric_field_id:
            if df[col].dtype == object:
                group_field_id = col
                break

    if numeric_field_id is not None:
        print(f"Selected numeric field for EDA: {numeric_field_id}")
        # Choose a threshold for demonstration
        if pd.api.types.is_integer_dtype(df[numeric_field_id]):
            threshold = df[numeric_field_id].quantile(0.95)
        else:
            threshold = df[numeric_field_id].mean()

        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"\nFiltered records where {numeric_field_id} > {threshold:.3f} (showing up to 5):")
        display(filtered_df.head())

        # Normalize the numeric field
        norm_col = numeric_field_id + "_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for the filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Group by group_field if available
        if group_field_id is not None:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped (mean) {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
    else:
        print("No numeric fields found for EDA in selected record set.")
else:
    print("No main record set DataFrame available for EDA.")

## 5. Visualization
Visualize the distribution of the selected numeric field and its groupings (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    
    if group_field_id is not None:
        plt.figure(figsize=(8,4))
        # Only plot first 10 groups for clarity
        order = df[group_field_id].value_counts().index[:10]
        sns.boxplot(y=df[numeric_field_id], x=df[group_field_id], order=order)
        plt.xticks(rotation=45, ha='right')
        plt.title(f"{numeric_field_id} by {group_field_id} (Top 10)")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.tight_layout()
        plt.show()

## 6. Conclusion

In this notebook, you used the `mlcroissant` Python library to access a FAIR² dataset defined by a Croissant schema. The workflow included:
- Loading and inspecting dataset metadata
- Listing available record sets and their field `@id`s
- Dynamically extracting all records into DataFrames with references by `@id`
- Automated demonstration of EDA steps (filtering, normalization, grouping)
- Visualization of distributions and group effects

**Note:** All references to fields and record sets use their Croissant `@id`. Refer to the schema documentation for precise `@id` meanings and for advanced analytics beyond this introductory workflow.

_Dataset source: https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json_
